# 🏗️ Notebook: Introduction to Retrieval-Augmented Generation (RAG)

In this notebook, we'll learn how to use **Retrieval-Augmented Generation (RAG)** to improve the responses of large language models (LLMs) with external knowledge sources.

## 📚 Sources

- [LangChain Documentation on RAG](https://python.langchain.com/docs/tutorials/rag/)


The functionality remains the same, but the code now follows current LangChain best practices.

---

In [2]:
import langchain
langchain.__version__

'1.1.3'

In [3]:
# Import necessary libraries
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

In [ ]:
LLM_URL = "http://localhost:11434"
LLM_MODEL = "gpt-oss:20b" 
EMBEDDING_MODEL = "nomic-embed-text" # The university's Ollama server provides us with an embedding model that we can use to convert documents into vectors.

In [5]:
# Initialize LLM
llm = ChatOllama(
    model=LLM_MODEL,
    base_url=LLM_URL,
    temperature=0
)

## 1. Load and Prepare Documents

We'll use a PDF document about the history of toxicology at the Technical University of Munich (TUM) as an example.

First, we'll follow three steps:

* Download the PDF from the official website (manual download is also possible!)
* Convert the PDF into text
* Split into chunks

In [6]:
path_pdf = "https://mediatum.ub.tum.de/doc/1770933/document.pdf"

# download as "tum_toxicology.pdf" if not already present
import os, requests
if not os.path.exists("content/tum_toxicology.pdf"): 
    response = requests.get(path_pdf)
    with open("content/tum_toxicology.pdf", "wb") as f:
        f.write(response.content)

### Load Documents

In LangChain, there are many ways to load documents. Depending on the file type (e.g., PDF, DOCX, TXT, HTML), there are different loader classes.

In our case, we use the `PyPDFLoader` to load the PDF file.

More examples of loaders:

```python
# 1. For PDFs
from langchain_community.document_loaders import PyPDFLoader
# 2. For text files / strings
from langchain_community.document_loaders import TextLoader
# 3. For web pages
from langchain_community.document_loaders import WebBaseLoader
# 4. For HTML files
from langchain_community.document_loaders import HTMLLoader
```

You can find more information about the different loaders in the [LangChain Documentation](https://docs.langchain.com/oss/javascript/integrations/document_loaders).

All loader classes have a `load()` method that reads the file and converts it into a list of `Document` objects.
Each `Document` object contains the text content.

In [7]:
# Load document
# More about the Document object (https://docs.langchain.com/oss/python/integrations/document_loaders).
loader = PyPDFLoader("content/tum_toxicology.pdf")
full_pdf = loader.load()  # load() returns a list of Document objects, one Document per page in the PDF

# Split text into chunks
# RecursiveCharacterTextSplitter divides long texts into smaller sections (chunks)
# This is important for RAG systems, as embedding models often have length limitations of e.g. 512 or 1024 tokens
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1024,        # Maximum length of a chunk in characters
    chunk_overlap=50,      # Overlap between chunks prevents information loss at boundaries
    separators=["\n", ",", " ", ""]       # Text is primarily split at paragraphs (double line breaks)
)
# split_documents() applies the splitting to the Document object
# and returns a list of smaller Document objects
docs = text_splitter.split_documents(full_pdf)

# page_content contains the actual text of a Document object
print(f"Number of chunks: {len(docs)}")
print(f"\nExample chunk:\n{docs[0].page_content}")

Number of chunks: 27

Example chunk:
Vol.:(0123456789)
Naunyn-Schmiedeberg's Archives of Pharmacology (2024) 397:9597–9602 
https://doi.org/10.1007/s00210-024-03315-0
REVIEW
History of toxicology at the Technical University of Munich (TUM)
Helmut Greim1
Received: 9 July 2024 / Accepted: 17 July 2024 / Published online: 24 July 2024 
© The Author(s) 2024
Abstract
Toxicology at the TUM is mainly associated with the Faculty of Medicine at the Klinikum rechts der Isar (MRI). The Depart-
ment of Clinical Toxicology has been founded in 1963. Max von Clarmann, the head, focused his activities on the treatment 
of intoxications and the development of analytical methods and established a poison information center. His successors, 
Thomas Zielker and Florian Eyer, further developed this department to an internationally renown institution.
In 1967, the MRI became the TUM faculty of medicine with its Institute of Pharmacology and Toxicology. The director


The `RecursiveCharacterTextSplitter` tries the separators from top to bottom in the specified order. It always takes the first separator that can split the text into chunks `≤` `chunk_size`. If a chunk is still too large, it tries the next separator in the list until the text is small enough.

The `chunk_overlap` ensures that consecutive chunks overlap. The last X characters of a chunk are repeated at the beginning of the next chunk, so that important information spanning chunk boundaries is not lost and the semantic context is preserved. [Here](https://dev.to/tak089/what-is-chunk-size-and-chunk-overlap-1hlj) you can find another explanation.

## 2. RAG with Keyword-based Search (BM25)

BM25 is a classic ranking algorithm based on **keyword matching**. It calculates how relevant a document is for a search query, based on the frequency and distribution of search terms in the chunks. BM25 calculates a relevance score for each document and sorts them accordingly.

In [8]:
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 10  # Number of documents to return

# Example search with the BM25 retriever
# .invoke() returns the 10 most relevant documents
query = "When was the Department of Clinical Toxicology founded?"
retrieved_docs = bm25_retriever.invoke(query)

### Uncomment to display the found documents: ###
# print(f"Query: {query}\n")
# print("Found documents:")
# for i, doc in enumerate(retrieved_docs):
#     print(f"\n--- Document {i+1} ---")
#     print(doc.page_content)

In [9]:
# Let's create a prompt with ChatPromptTemplate. ChatPromptTemplate allows us to create flexible prompts with placeholders.
# The placeholders will later be replaced with the actual values
template = """Answer the question based on the following context:

Context: {context}

Question: {question}

Answer:"""

# Function to format the retrieved documents into a string
def format_docs(docs):
    # All documents are concatenated into a string, separated by two line breaks
    return "\n\n".join(doc.page_content for doc in docs)

# The input_variables must match the placeholders in the template
prompt = ChatPromptTemplate.from_template(template)

# The RAG pipeline for BM25:
# 1. bm25_retriever: Retrieves the most relevant documents for the question based on keyword matching
# 2. RunnableLambda(format_docs): Converts the list of documents into a coherent string
# 3. RunnablePassthrough(): Passes the input (the question) directly without changing it. In our example, question_1
# 4. prompt: Inserts the formatted context and question into the prompt
# 5. llm: Generates the final answer based on the prompt. Prompt expects "context" and "question" as inputs, hence the dictionary at the beginning.
# "|": Chains Runnable objects in the LangChain Expression Language (LCEL), where the output of the left object becomes the input of the right.
qa_chain_bm25 = (
    {"context": bm25_retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()} 
    | prompt
    | llm
)

# Ask questions
question_1 = "Who founded the Department of Clinical Toxicology at TUM?"
result = qa_chain_bm25.invoke(question_1) # .invoke() executes the chain
print(f"Question: {question_1}")
print(f"Answer: {result.content}")

Question: Who founded the Department of Clinical Toxicology at TUM?
Answer: The Department of Clinical Toxicology at the Technical University of Munich was founded by **Max von Clarmann**.


## 3. RAG with Embeddings (Semantic Search)

In **embedding-based search**, texts are converted into numerical vectors that represent their semantic meaning.

We use ChromaDB (often referred to as "Chroma"), an open-source vector database. Each vector is associated with a text chunk, so that when we perform a search query, we can find the semantically most similar chunks.

With `vectorstore.get()` we could retrieve all vectors as a Python dictionary.

In [10]:
# Initialize embeddings model.
# The university's Ollama server has the "nomic-embed-text" model available, which we can use to convert texts into vectors.
# Important notes:
# (1) Converting to vectors takes time and computational resources. Depending on document length, this can take several minutes, so please be patient and prepare your data well!
# (2) The embedding model has a length limitation of about 512 tokens. Longer texts must be split into chunks beforehand!

ollama_embeddings = OllamaEmbeddings(
    model=EMBEDDING_MODEL,
    base_url=LLM_URL
)

try:
  # I added this code snippet to ensure that when running the notebook repeatedly, no duplicate vector store is created.
  vectorstore.delete_collection() 
except:
  pass

# Create a vector store from the documents with Chroma.
vectorstore = Chroma.from_documents(docs, ollama_embeddings)

# Create retriever. The parameter k specifies how many similar documents should be returned.
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Example search
query = "What research topics were investigated at the Institute of Toxicology?"
retrieved_docs = vector_retriever.invoke(query)

print(f"Query: {query}\n")
print("Found documents:")
for i, doc in enumerate(retrieved_docs[:1]):
    print(f"\n--- Document {i+1} ---")
    print(doc.page_content)

Query: What research topics were investigated at the Institute of Toxicology?

Found documents:

--- Document 1 ---
Toxicology was Melchior Reiter, formerly a professor at the 
LMU Institute of Pharmacology. At the same time, Reiter 
chaired a small unit of pharmacology at the GSF, a federal 
research institute in a suburb of Munich. There, Reiter rec-
ommended to establish an Institute of Toxicology with the 
focus on the potential adverse health effects of environmen-
tal chemicals. The institute rapidly expanded and became 
one of the leading research institutes of toxicology in Ger -
many with a great international reputation. More details 
about the foundation and activities of the different institutes 
involved in toxicology research and teaching are presented.
The department of (clinical) toxicology
Toxicology at the TUM is mainly associated with the Faculty 
of Medicine at the Klinikum rechts der Isar (MRI). Before 
the clinic became part of the TUM, the clinician Max von 
Clar

In [11]:
# We can now create a RAG chain with the embedding-based retriever.
qa_chain_embeddings = (
    {"context": vector_retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()}
    | prompt
    | llm
)

# Ask question
result = qa_chain_embeddings.invoke(query)
print(f"Question: {query}")
print(f"Answer: {result.content}")

Question: What research topics were investigated at the Institute of Toxicology?
Answer: The Institute of Toxicology was set up to study the **potential adverse health effects of environmental chemicals**. Its research focused on environmental toxicology – assessing how chemicals in the environment can harm human health, investigating mechanisms of toxicity, and evaluating chemical safety and risk.


## Exercise: Load Data from Multiple Text Files

In practice, you often don't want to use just a single document, but rather a whole collection of text files as a knowledge base for RAG.
Research how to load multiple text files from a directory using `DirectoryLoader`.

I have provided you with some text files via GRIPS that you can use for RAG.
These are short reports about solar eclipses in the coming decades.

Here's how you can create a RAG pipeline with these text files:

* 1. Load all text files from `datenbank/` with `DirectoryLoader`
* 2. Create a vector store with Chroma and Ollama embeddings
* 3. Use a `RecursiveCharacterTextSplitter` again to split the documents into chunks. Since our embedding model has a maximum input length of 512 tokens, we set the `chunk_size` to 500 and `chunk_overlap` to 50 to ensure that the chunks comply with the length limitation.
* 4. Create a `vector_retriever` that returns the 5 semantically most similar chunks for a search query.
* 5. Test the `vector_retriever` with an example search query, e.g., "When will there be a total solar eclipse in the polar regions?"
* 6. Create a RAG chain with the embedding-based retriever and test it with the same search query.

In [12]:
# Insert code here...

<details>
<summary><b>Show solution</b></summary>

<p>Load documents from multiple text files:</p>
<br/>
-------

```python
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "datenbank/",
    glob="*.txt", # Load only .txt files
    loader_cls=TextLoader
)

documents = loader.load()

# Important!: Although the documents in the database are usually already short, we still split them into chunks to ensure
# that they comply with the length limitations of the embedding models.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n", ",", " ", ""]
)

docs = text_splitter.split_documents(documents)
vectorstore.delete_collection()
vectorstore = Chroma.from_documents(docs, ollama_embeddings)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
```

<p>Test the `vector_retriever` with an example search query:</p>
<br/>
-------

```python
# Example search
query = "When will there be a total solar eclipse in the polar regions?"
retrieved_docs = vector_retriever.invoke(query)


qa_chain_embeddings = (
    {"context": vector_retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()}
    | prompt
    | llm
)

# Ask question
result = qa_chain_embeddings.invoke(query)
print(f"Answer: {result.content}")
```

</details>